# Hanssem Macro BigQuery ETL

운영자용 Colab 노트북입니다.

실행 순서:
1. 저장소 새로 받기 및 설치
2. Google 인증
3. 환경변수 설정
4. BigQuery 데이터마트 초기화
5. 전체 `run-bq` 또는 MOLIT 지표만 재실행
6. BigQuery 결과 확인

## 1. 저장소 새로 받기 및 설치

GitHub 최신 변경을 반영하려면 이 셀부터 실행하세요.

In [ ]:
%cd /content
!rm -rf hanssem-macro-dashboard
!git clone https://github.com/leeyj1545-lang/hanssem-macro-dashboard.git
%cd /content/hanssem-macro-dashboard
!pip install -r requirements.txt

## 2. Google 인증

In [ ]:
from google.colab import auth
auth.authenticate_user()

## 3. 운영자 설정값

아래 값만 수정하면 됩니다.

In [ ]:
import os
import sys

BQ_PROJECT_ID = "cellular-client-310600"
BQ_DATASET = "Yunjae_Workspace"
BQ_LOCATION = "US"

KOSIS_API_KEY = "YOUR_KOSIS_API_KEY"
RONE_API_KEY = "YOUR_RONE_API_KEY"
ECOS_API_KEY = ""  # optional
DATA_GO_KR_API_KEY = ""  # optional

os.environ["BQ_PROJECT_ID"] = BQ_PROJECT_ID
os.environ["BQ_DATASET"] = BQ_DATASET
os.environ["BQ_LOCATION"] = BQ_LOCATION
os.environ["BIGQUERY_PROJECT_ID"] = BQ_PROJECT_ID
os.environ["BIGQUERY_DATASET"] = BQ_DATASET
os.environ["BIGQUERY_LOCATION"] = BQ_LOCATION
os.environ["KOSIS_API_KEY"] = KOSIS_API_KEY
os.environ["RONE_API_KEY"] = RONE_API_KEY
os.environ["ECOS_API_KEY"] = ECOS_API_KEY
os.environ["DATA_GO_KR_API_KEY"] = DATA_GO_KR_API_KEY

sys.path.insert(0, "/content/hanssem-macro-dashboard/src")
print(f"Configured BigQuery target: {BQ_PROJECT_ID}.{BQ_DATASET} ({BQ_LOCATION})")

## 4. BigQuery 데이터마트 초기화

In [ ]:
from hanssem_macro_dashboard.bq import initialize_bigquery_datamart

initialize_bigquery_datamart()
print(f"Initialized BigQuery datamart: {BQ_PROJECT_ID}.{BQ_DATASET}")

## 5-A. 전체 운영 실행

전체 지표를 한 번에 실행합니다.

In [ ]:
!python -m hanssem_macro_dashboard.pipeline run-bq

## 5-B. MOLIT 지표만 재실행

국토교통 통계누리 파일 지표만 다시 시도할 때 사용합니다.

In [ ]:
!python -m hanssem_macro_dashboard.pipeline run-bq --indicator completion_volume --indicator housing_permits --indicator unsold_units

## 6. BigQuery 조회 헬퍼

In [ ]:
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client(project=BQ_PROJECT_ID)

def run_query(sql: str) -> pd.DataFrame:
    rows = [dict(row.items()) for row in client.query(sql, location=BQ_LOCATION).result()]
    return pd.DataFrame(rows)

## 7. ETL 실행 이력 확인

In [ ]:
sql = f"""
SELECT
  run_id,
  indicator_id,
  collect_status,
  stage_status,
  rows_loaded,
  latest_period,
  started_at,
  finished_at,
  message
FROM `{BQ_PROJECT_ID}.{BQ_DATASET}.etl_run_history`
ORDER BY started_at DESC, indicator_id
LIMIT 50
"""
run_query(sql)

## 8. Source Verification 확인

In [ ]:
sql = f"""
SELECT
  indicator_id,
  source_name,
  provider,
  verification_status,
  error_type,
  row_count,
  last_verified_at,
  message
FROM `{BQ_PROJECT_ID}.{BQ_DATASET}.source_verification`
ORDER BY indicator_id
"""
run_query(sql)

## 9. 적재된 indicator 수 확인

In [ ]:
sql = f"""
SELECT indicator_id, COUNT(*) AS cnt
FROM `{BQ_PROJECT_ID}.{BQ_DATASET}.macro_indicator_observations`
GROUP BY indicator_id
ORDER BY indicator_id
"""
run_query(sql)

## 10. HMI view 확인

In [ ]:
sql = f"""
SELECT
  date,
  price_index,
  jeonse_index,
  completion_volume,
  unsold_units,
  hmi,
  signal,
  market_phase,
  insight_text
FROM `{BQ_PROJECT_ID}.{BQ_DATASET}.vw_hanssem_macro_hmi`
ORDER BY date DESC
LIMIT 20
"""
run_query(sql)

## 참고

- 운영 기준 실행: `python -m hanssem_macro_dashboard.pipeline run-bq`
- MOLIT 지표만 재실행: `python -m hanssem_macro_dashboard.pipeline run-bq --indicator ...`
- BigQuery에서 5개 지표가 모두 보이면 Tableau 연결 단계로 이동합니다.